# Merge_Vehicle_Movements_To_Silver

Turns raw vehicle movement events from the Eventstream (`bronze_streaming.vehicle_movements`) into a clean, typed, de-duplicated Silver table (`silver_vehicle_movements`) that Gold and reporting rely on.

Runs first in `PL_ColdTrace_Vehicle_Telemetry`, followed by `Build_Gold_Vehicle_Tables`.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable
from notebookutils import mssparkutils
import json

bronze_table_name = "bronze_streaming.vehicle_movements"
silver_table_name = "silver_vehicle_movements"
LOOKBACK_HOURS = 1   # safety buffer for events that arrive late

if not spark.catalog.tableExists(bronze_table_name):
    raise ValueError(f"Table '{bronze_table_name}' was not found. Is the Eventstream running?")

In [ ]:
# Read Bronze and parse the string timestamps (Eventstream stores them with
# 7 fractional digits, cut to milliseconds here) into real timestamps.
timestamp_format = "yyyy-MM-dd'T'HH:mm:ss.SSS"
df_bronze = (
    spark.read.table(bronze_table_name)
    .withColumn("EventTimestamp", F.to_timestamp(F.substring(F.col("EventTimestamp").cast("string"), 1, 23), timestamp_format))
    .withColumn("PlannedArrival", F.to_timestamp(F.substring(F.col("PlannedArrival").cast("string"), 1, 23), timestamp_format))
)

# Incremental window: only events newer than the latest one already in
# Silver (minus a look-back buffer for late arrivals). Streaming has no
# files to track, so a time window replaces the file-name watermark used
# in the orders pipeline.
if spark.catalog.tableExists(silver_table_name):
    last_event_time = spark.sql(f"SELECT MAX(EventTimestamp) AS last_ts FROM {silver_table_name}").collect()[0].last_ts
else:
    last_event_time = None

if last_event_time is not None:
    df_bronze = df_bronze.filter(
        F.col("EventTimestamp") >= F.expr(f"timestamp'{last_event_time}' - interval {LOOKBACK_HOURS} hours")
    )

incoming_count = df_bronze.count()
if incoming_count == 0:
    mssparkutils.notebook.exit(json.dumps({"status": "no_new_events", "rows": 0}))

In [ ]:
# Clean, type and identify events. VehicleState is extracted from the
# Properties JSON (handles struct, JSON string or missing); rows missing an
# identifying field are dropped since they can't be placed in time.
# MovementId is a deterministic hash over the identifying fields, used for
# de-duplication and as the merge key.
properties_type = dict(df_bronze.dtypes).get("Properties", None)
if properties_type and properties_type.startswith("struct"):
    vehicle_state = F.col("Properties.BusState")
elif properties_type == "string":
    vehicle_state = F.get_json_object("Properties", "$.BusState")
else:
    vehicle_state = F.lit(None).cast("string")

df_typed = (
    df_bronze
    .withColumn("VehicleState", vehicle_state)
    .withColumn("VehicleTripId", F.col("VehicleTripId").cast("string"))
    .withColumn("RouteId", F.col("RouteId").cast("string"))
    .withColumn("CheckpointId", F.col("CheckpointId").cast("int"))
    .select("VehicleTripId", "RouteId", "CheckpointId", "PlannedArrival", "EventTimestamp", "VehicleState")
    .filter(
        F.col("VehicleTripId").isNotNull() & F.col("RouteId").isNotNull() &
        F.col("CheckpointId").isNotNull() & F.col("EventTimestamp").isNotNull()
    )
    .withColumn(
        "MovementId",
        F.sha2(F.concat_ws("|", "VehicleTripId", "CheckpointId", "EventTimestamp",
                           F.coalesce("VehicleState", F.lit(""))), 256)
    )
)

# Remove duplicates inside the current batch.
window_by_movement = Window.partitionBy("MovementId").orderBy(F.col("EventTimestamp").desc())
df_incoming = (
    df_typed.withColumn("row_num", F.row_number().over(window_by_movement))
    .filter(F.col("row_num") == 1).drop("row_num")
    .withColumn("ProcessedAt", F.current_timestamp())
)

In [ ]:
# Write to Silver: first run creates the table; later runs merge on
# MovementId, inserting only unknown IDs — events are immutable, so
# existing rows are never updated. Re-running the notebook is safe (idempotent).
if spark.catalog.tableExists(silver_table_name):
    (DeltaTable.forName(spark, silver_table_name).alias("target")
     .merge(df_incoming.alias("source"), "target.MovementId = source.MovementId")
     .whenNotMatchedInsertAll()
     .execute())
    print(f"Delta merge completed successfully into '{silver_table_name}'.")
else:
    df_incoming.write.format("delta").mode("overwrite").saveAsTable(silver_table_name)
    print(f"Silver table '{silver_table_name}' initialized.")

mssparkutils.notebook.exit(json.dumps({"status": "silver_merge_success", "incoming_rows": incoming_count}))